In [0]:
# CELL 1: Setup & Load
# -------------------------------------------------------------------------
from pyspark.sql.functions import col, to_date, when, lit
 
# 1. Define Azure Config (PASTE YOUR KEYS if needed)
storage_account_name = "adlsdubaipreethi"
# Note: spark.conf.set for fs.azure.account.key is not supported on serverless compute.
# Access relies on Unity Catalog external locations / storage credentials.
 
# 2. Define Paths
base_path = f"abfss://bronze@adlsdubaipreethi.dfs.core.windows.net"
#/bronze/projects/landing/current"
silver_path = f"abfss://silver@adlsdubaipreethi.dfs.core.windows.net"
 
# 3. Read Bronze CSVs
print("Reading Bronze files...")
df_trans_bronze = spark.read.option("header", "true").csv(f"{base_path}/transactions/landing/current")
df_proj_bronze = spark.read.option("header", "true").csv(f"{base_path}/projects/landing/current")
df_units_bronze = spark.read.option("header", "true").csv(f"{base_path}/units/landing/current")
 
print("✅ Bronze Data Loaded & Silver Container Configured.")

In [0]:
# CELL 2: Clean Projects Dimension (Status & Dates Only)
# -------------------------------------------------------------------------
from pyspark.sql.functions import expr

df_projects_clean = df_proj_bronze.select(
# try_cast / try_to_date return NULL for malformed values such as the string 'null'
expr("try_cast(project_number AS INT)").alias("project_number"),
col("project_status"),
expr("try_cast(percent_completed AS DOUBLE)").alias("percent_completed"),
 
# FIX: Date format set to 'dd-MM-yyyy'
expr("try_to_date(project_start_date, 'dd-MM-yyyy')").alias("start_date"),
expr("try_to_date(completion_date, 'dd-MM-yyyy')").alias("completion_date"),
 
col("master_project_en").alias("master_community")
).dropDuplicates(["project_number"])
 
print("✅ Projects Cleaned.")
#display(df_projects_clean.limit(5))

In [0]:
display(df_projects_clean.limit(5))

In [0]:
# CELL 3: Clean Transactions Fact
# -------------------------------------------------------------------------
# Re-read: transactions/landing/current does not exist; CSV is at transactions/landing/
df_trans_bronze = spark.read.option("header", "true").csv(
    f"abfss://bronze@adlsdubaipreethi.dfs.core.windows.net/transactions/landing/current.csv")

df_trans_clean = df_trans_bronze.select(
col("transaction_id"),
 
# FIX: Date format set to 'dd-MM-yyyy'; try_to_date tolerates 'null' strings
expr("try_to_date(instance_date, 'dd-MM-yyyy')").alias("transaction_date"),
 
col("procedure_name_en").alias("transaction_type"),
col("trans_group_en").alias("transaction_group"),
 
# Location & Project Info
col("area_name_en").alias("area_name"),
col("building_name_en").alias("building_name"),
expr("try_cast(project_number AS INT)").alias("project_number"),
 
# --- We get the English Name from here ---
col("project_name_en").alias("project_name"),
col("nearest_landmark_en").alias("landmark"),
 
# Property Details
col("property_type_en").alias("property_type"),
col("property_sub_type_en").alias("property_subtype"),
col("rooms_en").alias("room_type"), # e.g., "1 B/R"
 
# Metrics
expr("try_cast(procedure_area AS DOUBLE)").alias("area_sqft"),
expr("try_cast(actual_worth AS DOUBLE)").alias("price"),
expr("try_cast(meter_sale_price AS DOUBLE)").alias("meter_sale_price")
)
 
# Filter garbage data
df_trans_clean = df_trans_clean.filter(col("price") > 0)
 
print("✅ Transactions Cleaned.")
display(df_trans_clean.limit(5))

In [0]:
# CELL 4: Join & Enrich (Silver Master Table)
# -------------------------------------------------------------------------
 
# 1. Join Transactions with Projects
df_enriched = df_trans_clean.join(
df_projects_clean,
on="project_number",
how="left"
)
 
# 2. Enrich: Calculate Price Per SqFt
df_enriched = df_enriched.withColumn(
"price_per_sqft",
when(col("area_sqft") > 0, col("price") / col("area_sqft")).otherwise(0)
)
 
# 3. Enrich: Is Off-Plan?
df_enriched = df_enriched.withColumn(
"is_offplan_flag",
when(
(col("transaction_date") < col("completion_date")) | (col("percent_completed") < 100),
lit(True)
).otherwise(lit(False))
)
 
print("✅ Data Enriched & Joined.")
display(df_enriched.select("transaction_date", "project_name", "room_type", "price", "is_offplan_flag").limit(10))


In [0]:
# CELL 5: Write to Silver Layer (Physical Files)
# -------------------------------------------------------------------------
 
# 1. Write the Master Transactions Table
print("Writing Transactions to Silver...")
df_enriched.write.format("delta").mode("overwrite").save(f"{silver_path}/transactions")
 
# 2. Write the Clean Units Table
print("Writing Units to Silver...")
df_units_clean = df_units_bronze.select(
col("property_id"), # Unique Key
expr("try_cast(project_id AS INT)").alias("project_number"), # Rename to match other tables
col("unit_number"),
 
# Metrics
col("rooms").alias("bedrooms"), # Numeric (e.g., "1")
col("rooms_en").alias("room_type"),# Text (e.g., "1 B/R")
 
# --- FIX: Use 'actual_area' ---
expr("try_cast(actual_area AS DOUBLE)").alias("unit_area")
 
).dropDuplicates(["property_id"]) # Deduplicate
 
df_units_clean.write.format("delta").mode("overwrite").save(f"{silver_path}/units")
 
print("🎉 SUCCESS: Silver Delta files written to Azure container 'silver'.")